### The objective of this notebook is to apply hierarchical reconciliation through MinTrace (Minimum Trace) method to the base forecasts, so as to ensure that the forecasts sum coherently across the hierarchy (Route -> Airport -> Total) 

In [2]:
# Importing the libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from statsforecast import StatsForecast
from statsforecast.models import Naive, SeasonalNaive, AutoARIMA, AutoETS

from hierarchicalforecast.core import HierarchicalReconciliation
from hierarchicalforecast.methods import MinTrace
from hierarchicalforecast.utils import aggregate

In [3]:
# Load the full processed dataset
df_full = pd.read_csv('../data/processed/hierarchy_ready.csv')
df_full['ds'] = pd.to_datetime(df_full['ds'])

In [4]:
# Dropping the 4 missing series
to_drop = {
    'TOTAL/MUMBAI/MUMBAI-RAJKOT',
    'TOTAL/DELHI/DELHI-RAJKOT',
    'TOTAL/DEHRA DUN/DEHRA DUN-DELHI',
    'TOTAL/DEHRA DUN'
}

df_full = df_full[~df_full['unique_id'].isin(to_drop)].copy()

In [5]:
# Train Test Split (Same as that used for forecasting)

split_date = pd.Timestamp('2025-03-01')
train = df_full[df_full['ds'] < split_date].copy()
test = df_full[df_full['ds'] >= split_date].copy()

In [6]:
# Loading the baseforecasts

base_forecasts = pd.read_csv('../outputs/base_forecasts.csv')
base_forecasts['ds'] = pd.to_datetime(base_forecasts['ds'])

In [8]:
print('Full data: ',df_full.shape, ' | Series: ',df_full['unique_id'].nunique())
print("Train: ",train.shape, " | Test: ",test.shape)
print('Base Forecasts: ',base_forecasts.shape)
print('Models in base forecasts: ',[c for c in base_forecasts.columns if c not in ['ds','unique_id']])

Full data:  (26677, 3)  | Series:  228
Train:  (23941, 3)  | Test:  (2736, 3)
Base Forecasts:  (2736, 6)
Models in base forecasts:  ['Naive', 'SeasonalNaive', 'AutoARIMA', 'AutoETS']


In [9]:
# Building the hierarchy structure

# Preparing data for the aggregate function

# Filter for route series

routes_only = df_full[df_full['unique_id'].str.count('/') == 2].copy()

# Extract the origin and destination for each unique id

routes_only['origin'] = routes_only['unique_id'].str.split('/').str[1]

routes_only['route_label'] = routes_only['unique_id'].str.split('/').str[2]

print("Route rows: ", routes_only.shape)

print("Unique routes: ",routes_only['route_label'].nunique())

print('Unique origin airports: ',routes_only['origin'].nunique())

print(routes_only[['unique_id','origin','route_label','ds','y']].head())


Route rows:  (22955, 5)
Unique routes:  197
Unique origin airports:  30
                            unique_id    origin  ...         ds        y
213  TOTAL/AGARTALA/AGARTALA-GUWAHATI  AGARTALA  ... 2019-05-01  11338.0
214  TOTAL/AGARTALA/AGARTALA-GUWAHATI  AGARTALA  ... 2019-06-01  10786.0
215  TOTAL/AGARTALA/AGARTALA-GUWAHATI  AGARTALA  ... 2019-07-01  11355.0
216  TOTAL/AGARTALA/AGARTALA-GUWAHATI  AGARTALA  ... 2019-08-01  10927.0
217  TOTAL/AGARTALA/AGARTALA-GUWAHATI  AGARTALA  ... 2019-09-01   9271.0

[5 rows x 5 columns]


In [19]:
# Define the hierarchy specification
routes_for_agg = routes_only.drop(columns = ['unique_id']).copy()
routes_for_agg['total'] = 'TOTAL'
spec = [
    ['total'],
    ['total','origin'],
    ['total','origin','route_label']
]

# Build the hierarchy
Y_df, S_df, tags = aggregate(routes_for_agg, spec)

print('Y_df (long-format reconstructed hierarchy): ', Y_df.shape)
print('S_df (the S matrix): ',S_df.shape)
print('Tags (groups of series at each level): ',{k:len(v) for k, v in tags.items()})
print('S matrix preview: ')
print(S_df.head())

Y_df (long-format reconstructed hierarchy):  (26677, 3)
S_df (the S matrix):  (228, 198)
Tags (groups of series at each level):  {'total': 1, 'total/origin': 30, 'total/origin/route_label': 197}
S matrix preview: 
         unique_id  ...  TOTAL/NAGPUR/NAGPUR-PUNE
0            TOTAL  ...                       1.0
1   TOTAL/AGARTALA  ...                       0.0
2  TOTAL/AHMEDABAD  ...                       0.0
3     TOTAL/AIZAWL  ...                       0.0
4   TOTAL/AMRITSAR  ...                       0.0

[5 rows x 198 columns]


In [20]:
# Compute in-sample residuals

Y_df['ds'] = pd.to_datetime(Y_df['ds'])

# Train test split on Y_df
split_date = pd.Timestamp('2025-03-01')
Y_train = Y_df[Y_df['ds'] < split_date].copy()
Y_test = Y_df[Y_df['ds']>=split_date].copy()

# Refit the same 4 models on Y_train
models = [
    Naive(),
    SeasonalNaive(season_length = 12),
    AutoARIMA(season_length = 12),
    AutoETS(season_length = 12)
]

sf = StatsForecast(models = models, freq = 'MS', n_jobs = -1, verbose = True)
sf.fit(df = Y_train)

# Generate forecasts with fitted values = True to also get in-sample fitted values
fcst_with_insample = sf.forecast(df = Y_train, h = 12, fitted = True)

print('Forecast Shape: ',fcst_with_insample.shape)
print('Columns: ',fcst_with_insample.columns.tolist())

Forecast:   0%|          | 0/228 [Elapsed: 00:00]

Forecast Shape:  (2736, 6)
Columns:  ['unique_id', 'ds', 'Naive', 'SeasonalNaive', 'AutoARIMA', 'AutoETS']


In [27]:
# Get in-sample fitted values (training period predictions per model)

fitted_vals = sf.forecast_fitted_values()

print('Fitted values shape: ',fitted_vals.shape)
print('Columns: ',fitted_vals.columns.tolist())
print('Sample: ')
print(fitted_vals.head())

Fitted values shape:  (23941, 7)
Columns:  ['unique_id', 'ds', 'y', 'Naive', 'SeasonalNaive', 'AutoARIMA', 'AutoETS']
Sample: 
  unique_id         ds          y  ...  SeasonalNaive     AutoARIMA       AutoETS
0     TOTAL 2015-04-01  2329936.0  ...            NaN  2.327606e+06  2.326358e+06
1     TOTAL 2015-05-01  2512608.0  ...            NaN  2.334287e+06  2.329936e+06
2     TOTAL 2015-06-01  2296222.0  ...            NaN  2.550999e+06  2.512590e+06
3     TOTAL 2015-07-01  2401175.0  ...            NaN  2.239675e+06  2.296244e+06
4     TOTAL 2015-08-01  2352577.0  ...            NaN  2.437065e+06  2.401165e+06

[5 rows x 7 columns]


In [28]:
# Apply MinTrace Reconciliation

# Set up reconciliation engine with two MinTrace variants
reconcilers = [
    MinTrace(method = 'ols'),
    MinTrace(method = 'mint_shrink')
]

hrec = HierarchicalReconciliation(reconcilers = reconcilers)

# Apply the reconciliation
Y_rec_df = hrec.reconcile(
    Y_hat_df = fcst_with_insample,
    Y_df = fitted_vals,
    S_df= S_df,
    tags = tags
)

print('Reconciled Forecasts shape: ',Y_rec_df.shape)
print('Columns: ',Y_rec_df.columns.tolist())
print('Sample: ')
print(Y_rec_df.head())

Reconciled Forecasts shape:  (2736, 14)
Columns:  ['unique_id', 'ds', 'Naive', 'SeasonalNaive', 'AutoARIMA', 'AutoETS', 'Naive/MinTrace_method-ols', 'SeasonalNaive/MinTrace_method-ols', 'AutoARIMA/MinTrace_method-ols', 'AutoETS/MinTrace_method-ols', 'Naive/MinTrace_method-mint_shrink', 'SeasonalNaive/MinTrace_method-mint_shrink', 'AutoARIMA/MinTrace_method-mint_shrink', 'AutoETS/MinTrace_method-mint_shrink']
Sample: 
  unique_id  ... AutoETS/MinTrace_method-mint_shrink
0     TOTAL  ...                        5.777137e+06
1     TOTAL  ...                        5.393259e+06
2     TOTAL  ...                        5.013512e+06
3     TOTAL  ...                        4.989821e+06
4     TOTAL  ...                        5.067028e+06

[5 rows x 14 columns]


In [29]:
# Coherence Sanity check

check_month = '2025-12-01'
check_airport = 'TOTAL/MUMBAI'
model_col = 'AutoARIMA/MinTrace_method-mint_shrink'

# Check 1: TOTAL = Sum of Airport forecasts?
total_val = Y_rec_df[
    (Y_rec_df['unique_id'] == 'TOTAL') &
    (Y_rec_df['ds'] == check_month)][model_col].values[0] 

airport_sum = Y_rec_df[
    (Y_rec_df['unique_id'].str.count('/') == 1) &
    (Y_rec_df['ds'] == check_month)][model_col].sum()

print(f"Check 1: TOTAL vs. Sum of Airports for the month {check_month}")
print(f'TOTAL: {total_val:,.0f}' )
print(f'Sum of Airports: {airport_sum:,.0f}')
print(f'Match: {abs(total_val - airport_sum) < 1}')

Check 1: TOTAL vs. Sum of Airports for the month 2025-12-01
TOTAL: 5,912,429
Sum of Airports: 5,912,429
Match: True


In [30]:
# Check 2: Airport = Sum of all route forecasts?

mumbai_val = Y_rec_df[
    (Y_rec_df['unique_id'] == check_airport) &
    (Y_rec_df['ds'] == check_month)][model_col].values[0]

mumbai_routes_sum = Y_rec_df[
    (Y_rec_df['unique_id'].str.startswith(f'{check_airport}/')) &
    (Y_rec_df['ds'] == check_month)][model_col].sum()

print(f"Check 2: Airport vs. Sum of Routes for Mumbai for the month {check_month}")
print(f'Airport: {mumbai_val:,.0f}' )
print(f'Sum of Routes: {mumbai_routes_sum:,.0f}')
print(f'Match: {abs(mumbai_val - mumbai_routes_sum) < 1}')


Check 2: Airport vs. Sum of Routes for Mumbai for the month 2025-12-01
Airport: 377,118
Sum of Routes: 377,118
Match: True


In [32]:
# Save reconciled forecasts
Y_rec_df.to_csv('../outputs/reconciled_forecasts.csv',index = False)

# Save fitted values
fitted_vals.to_csv('../outputs/fitted_values.csv', index = False)

print("Saved both files")

Saved both files


In [33]:
# Merge reconciled forecasts with actuals from Y_test
eval_rec = Y_test.merge(Y_rec_df, on = ['unique_id','ds'], how = 'left')

# Identify the forecast columns
base_cols = ['Naive','SeasonalNaive', 'AutoARIMA','AutoETS']

rec_cols = [c for c in eval_rec.columns if 'MinTrace' in c]

print("Evaluation dataset shape: ",eval_rec.shape)
print("Base model columns: ", base_cols)
print("Reconciled model columns: ",rec_cols)
print("Total forecast columns: ",len(base_cols) + len (rec_cols))
print("Missing values per forecast column: ")
print(eval_rec[base_cols + rec_cols].isna().sum())


Evaluation dataset shape:  (2736, 15)
Base model columns:  ['Naive', 'SeasonalNaive', 'AutoARIMA', 'AutoETS']
Reconciled model columns:  ['Naive/MinTrace_method-ols', 'SeasonalNaive/MinTrace_method-ols', 'AutoARIMA/MinTrace_method-ols', 'AutoETS/MinTrace_method-ols', 'Naive/MinTrace_method-mint_shrink', 'SeasonalNaive/MinTrace_method-mint_shrink', 'AutoARIMA/MinTrace_method-mint_shrink', 'AutoETS/MinTrace_method-mint_shrink']
Total forecast columns:  12
Missing values per forecast column: 
Naive                                        0
SeasonalNaive                                0
AutoARIMA                                    0
AutoETS                                      0
Naive/MinTrace_method-ols                    0
SeasonalNaive/MinTrace_method-ols            0
AutoARIMA/MinTrace_method-ols                0
AutoETS/MinTrace_method-ols                  0
Naive/MinTrace_method-mint_shrink            0
SeasonalNaive/MinTrace_method-mint_shrink    0
AutoARIMA/MinTrace_method-mint_shri

In [34]:
# function for RMSE
def rmse (y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred)**2))

# function for MASE
def mase (y_true, y_pred, y_train, season_length = 12):
    naive_errors = np.abs(y_train[season_length:] - y_train[:-season_length])
    scale = naive_errors.mean()
    if scale == 0:
        return np.nan
    return np.mean(np.abs(y_true - y_pred)) / scale

# Building training history per series using Y_train so identifiers match Y_rec_df

train_history = Y_train.groupby('unique_id')['y'].apply(lambda s: s.values).to_dict()

# All 12 forecast columns to evaluate

all_forecast_cols = base_cols + rec_cols

# Compute metrics for every series-forecast pair

records = []
for uid, g in eval_rec.groupby('unique_id'):
    y_true = g['y'].values
    y_train_series = train_history[uid]
    for col in all_forecast_cols:
        y_pred = g[col].values
        records.append({
            'unique_id':uid,
            'forecast':col,
            'rmse':rmse(y_true, y_pred),
            'mase':mase(y_true, y_pred, y_train_series, season_length = 12)
        })

metrics_rec_df = pd.DataFrame(records)

print("Metrics computed: ", len(metrics_rec_df), " rows")
print("Sample: ")
print(metrics_rec_df.head(12))

Metrics computed:  2736  rows
Sample: 
   unique_id  ...      mase
0      TOTAL  ...  0.240665
1      TOTAL  ...  0.287239
2      TOTAL  ...  0.229801
3      TOTAL  ...  0.240671
4      TOTAL  ...  0.240665
5      TOTAL  ...  0.287239
6      TOTAL  ...  0.230091
7      TOTAL  ...  0.240637
8      TOTAL  ...  0.240665
9      TOTAL  ...  0.287239
10     TOTAL  ...  0.204525
11     TOTAL  ...  0.331970

[12 rows x 4 columns]


In [38]:
# Aggregate the comparison table
def aggregate_forecast(name):
    if '/' in name:
        model, method = name.split('/')
        method = method.replace('MinTrace_method-','') # ols or mint_shrink
    
    else:
        model = name
        method = 'base'
    
    return model, method

metrics_rec_df['model'] = metrics_rec_df['forecast'].apply(lambda x: aggregate_forecast(x)[0])
metrics_rec_df['method'] = metrics_rec_df['forecast'].apply(lambda x: aggregate_forecast(x)[1])

# Tag hierarchy level
metrics_rec_df['level'] = metrics_rec_df['unique_id'].apply(
    lambda x: 'TOTAL' if '/' not in x
    else ('Airport' if x.count('/') == 1 else 'Route')
)

# Pivot Summary 
summary = metrics_rec_df.groupby(['model','method','level'])['mase'].mean().round(3).reset_index()

mase_pivot = summary.pivot_table(index = 'model', columns = ['method','level'], values = 'mase')

mase_pivot = mase_pivot[['base','ols','mint_shrink']]

mase_pivot = mase_pivot.reindex(columns = pd.MultiIndex.from_product(
    [['base','ols','mint_shrink'], ['TOTAL','Airport','Route']]
))

print('========== Average MASE: Base vs Reconciled, By Model and Level ==========')
print(mase_pivot.to_string())

========== Average MASE: Base vs Reconciled, By Model and Level ==========
                base                   ols                mint_shrink               
               TOTAL Airport  Route  TOTAL Airport  Route       TOTAL Airport  Route
model                                                                               
AutoARIMA      0.230   0.618  0.755  0.230   0.713  0.837       0.205   0.594  0.801
AutoETS        0.241   0.599  0.756  0.241   0.599  0.808       0.332   0.594  0.842
Naive          0.241   0.603  0.799  0.241   0.603  0.799       0.241   0.603  0.799
SeasonalNaive  0.287   0.644  0.855  0.287   0.644  0.855       0.287   0.644  0.855


In [39]:
# Save the per-series metrics
metrics_rec_df.to_csv('../outputs/reconciled_metrics.csv', index = False)

# Save the aggregated metrics table
mase_pivot.to_csv('../outputs/reconciled_summary.csv', index = False)

print ("Saved both files")

Saved both files


In [42]:
# Coherence verification

all_cols = base_cols + rec_cols

# Check 1: Total = sum of airports for every (month x forecast)

total_check = []
for col in all_cols:
    
    total_vals = Y_rec_df[Y_rec_df['unique_id'] == 'TOTAL'][['ds',col]].rename(columns = {col:'total'})

    airport_vals = Y_rec_df[Y_rec_df['unique_id'].str.count('/') == 1].groupby('ds')[col].sum().reset_index().rename(columns = {col: 'airport_sum'})

    merged = total_vals.merge(airport_vals, on = 'ds')

    max_diff = (merged['total'] - merged['airport_sum']).abs().max()
    total_check.append({'forecast':col,'max_total_vs_airports_diff':max_diff})

total_check_df = pd.DataFrame(total_check)

print('===== TOTAL = sum of airports - max absolute difference per forecast =====')
print(total_check_df.to_string(index = False))

===== TOTAL = sum of airports - max absolute difference per forecast =====
                                 forecast  max_total_vs_airports_diff
                                    Naive                0.000000e+00
                            SeasonalNaive                0.000000e+00
                                AutoARIMA                1.494974e+05
                                  AutoETS                3.290130e+04
                Naive/MinTrace_method-ols                0.000000e+00
        SeasonalNaive/MinTrace_method-ols                2.793968e-09
            AutoARIMA/MinTrace_method-ols                6.519258e-09
              AutoETS/MinTrace_method-ols                3.725290e-09
        Naive/MinTrace_method-mint_shrink                9.313226e-10
SeasonalNaive/MinTrace_method-mint_shrink                1.862645e-09
    AutoARIMA/MinTrace_method-mint_shrink                4.656613e-09
      AutoETS/MinTrace_method-mint_shrink                4.656613e-09


In [44]:
# Check 2: Each airport = sum of its routes, for every (month x forecast)

airport_check = []
airports = [u for u in Y_rec_df['unique_id'].unique() if u.count('/') == 1]

for col in all_cols:
    max_diff_overall = 0
    for ap in airports:
        ap_vals = Y_rec_df[Y_rec_df['unique_id'] == ap][['ds',col]].rename(columns = {col:'airport'})
        
        route_vals = Y_rec_df[Y_rec_df['unique_id'].str.startswith(f'{ap}/')].groupby('ds')[col].sum().reset_index().rename(columns = {col:'route_sum'})
        
        merged = ap_vals.merge(route_vals,on = 'ds')

        max_difference_ap = (merged['airport'] - merged['route_sum']).abs().max()

        if max_difference_ap > max_diff_overall:
            max_diff_overall = max_difference_ap
        
    airport_check.append({'forecast':col, 'max_airport_vs_route_diff':max_diff_overall})

airport_check_df = pd.DataFrame(airport_check)

print('===== AIRPORT = sum of routes - max absolute difference per forecast (across all airports) =====')
print(airport_check_df.to_string(index = False))

===== AIRPORT = sum of routes - max absolute difference per forecast (across all airports) =====
                                 forecast  max_airport_vs_route_diff
                                    Naive               0.000000e+00
                            SeasonalNaive               0.000000e+00
                                AutoARIMA               8.700275e+04
                                  AutoETS               4.878129e+04
                Naive/MinTrace_method-ols               2.328306e-10
        SeasonalNaive/MinTrace_method-ols               6.984919e-10
            AutoARIMA/MinTrace_method-ols               4.656613e-10
              AutoETS/MinTrace_method-ols               4.656613e-10
        Naive/MinTrace_method-mint_shrink               2.910383e-11
SeasonalNaive/MinTrace_method-mint_shrink               4.656613e-10
    AutoARIMA/MinTrace_method-mint_shrink               4.656613e-10
      AutoETS/MinTrace_method-mint_shrink               4.656613e-10


### Observations and Inferences:

**Accuracy of forecasts:** AutoARIMA benefits the most from reconciliation at the Total and Airport levels. The MASE score improved from 0.230 to 0.205 with mint_shrink at Total level. However, at the route levels, there is a visible degradation after reconciliation. This is an expected trade-off in the process. SeasonalNaive and Naive forecasts remain the same after reconciliation. This is expected for structurally coherent forecasts. The AutoETS forecasts degraded significantly at TOTAL level with MASE score dropping to 0.332 from 0.241 after reconciliation. 


**Coherence of forecasts:** All reconciled forecasts were coherent to machine precision. Base AutoARIMA showed the highest mismatch between Total and Airport forecasts and airport and route forecasts which was nullified after reconciliation.

**Interpretation of Results:** 
Coherence was enforced via reconciliation and this is a pre-requisite in business problems like capacity planning, budget allocation. There is a trade-off in accuracy and it is asymmetric. The top two levels of hierarchy gains via reconciliation whereas the bottom suffers. The right choice of method for reconciliation depends on the priority of the level under consideration for that particular use case. 